# Demo — Agente Auditor Comunitario (AAC)

Ejemplos del agente **LangGraph** (Evaluación N°2). Cada celda se puede ejecutar sola con **Run** (Shift+Enter).

**Antes de empezar**

1. Entorno virtual activado (kernel **"Python (AAC)"** en VS Code).
2. Base indexada: `python -m src.indexer.build_index` (solo la primera vez).
3. API key de Groq opcional: **sin key el notebook corre igual en modo demo** (respuestas simuladas; el grafo, la memoria y las herramientas son reales). Con key válida en `.env` las respuestas son reales.

Cada celda hace trabajo real del agente: tarda ~20–30 s por ejemplo (menos en modo demo).

In [ ]:
import os
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv

load_dotenv()

from src.agent.llm import modo_demo
from src.grafo.grafo import invocar_agente

if modo_demo():
    print("MODO DEMO: sin API key valida -> el LLM se simula (grafo, memoria y RAG reales).")
else:
    print("API key detectada: el agente usa Groq (respuestas reales).")

In [ ]:
from IPython.display import Markdown, display


def mostrar(estado: dict, con_bitacora: bool = True) -> None:
    """Imprime lo esencial de una ejecución del agente y muestra la respuesta final."""
    plan = " -> ".join(p["nombre"] for p in estado.get("plan", []))
    aud = estado.get("auditoria") or {}
    red = estado.get("redaccion") or {}
    print(f"Tipo de tarea  : {estado.get('tipo_tarea')}")
    print(f"Plan priorizado: {plan}")
    print(f"Veredicto      : {aud.get('estado')} ({len(aud.get('alertas', []))} alertas)")
    print(f"Modo redaccion : {red.get('modo')}")
    print(f"Experiencias previas recuperadas: {len(estado.get('experiencias_previas', []))}")
    if estado.get("errores"):
        print(f"Errores        : {estado['errores']}")
    if con_bitacora:
        print("\nBitacora de decisiones:")
        for b in estado.get("bitacora", []):
            print(f"  [{b['nodo']}] {b['decision']}")
    print()
    display(Markdown(estado.get("respuesta_final", "")))


print("Helper mostrar() listo.")

## 1. Auditoría de un trabajo concreto (IE1, IE5)

Publicación de electricidad: el planificador clasifica `auditoria` y arma el plan priorizado
`memoria -> auditoria -> redaccion -> guardar_memoria`.

In [ ]:
estado = invocar_agente(
    "Necesito instalar un porton electrico nuevo en el acceso principal del edificio",
    categoria="electricidad",
    thread_id="demo-1",
)
mostrar(estado)

## 2. Publicación vaga — decisión adaptativa (IE7)

Con un texto sin detalle, el redactor **cambia de modo**: en vez de emitir un
informe de incumplimiento (o peor, inventar un problema), entrega un **cuestionario
y una plantilla con campos `[COMPLETAR]`** para que el administrador publique con
los datos que faltan. Prohibido inventar hechos o contenido legal.

In [ ]:
estado = invocar_agente(
    "Necesito ayuda con unas cosas del edificio",
    categoria="general",
    thread_id="demo-2",
)
mostrar(estado)

## 3. Consulta normativa — ruta alterna sin auditoría (IE7)

Una pregunta sobre la ley no dispara auditoría: el planificador la clasifica
`consulta_normativa` y la respuesta se redacta con citas.

In [ ]:
estado = invocar_agente(
    "¿Que obligaciones establece la Ley 21.442 sobre mantencion de ascensores en areas comunes?",
    categoria="ascensor",
    thread_id="demo-3",
)
mostrar(estado)

## 4. Memoria de corto plazo (IE3) — mismo `thread_id`, dos turnos

El checkpointer de LangGraph conserva el estado entre turnos: el segundo turno
parte con el historial del primero (se repite el mismo `thread_id`).

In [ ]:
turno1 = invocar_agente(
    "Necesito arreglar el porton electrico del edificio",
    categoria="electricidad",
    thread_id="demo-memoria-cp",
)
print("=== TURNO 1 ===")
mostrar(turno1, con_bitacora=False)

In [ ]:
turno2 = invocar_agente(
    "La misma obra, ahora con mas detalle: el porton es basculante de 400 kg y no tiene botonera interior",
    categoria="electricidad",
    thread_id="demo-memoria-cp",
)
print("=== TURNO 2 (mismo thread_id) ===")
mostrar(turno2, con_bitacora=False)
print("Historial acumulado del thread:", len(turno2.get("historial", [])), "entradas")

## 5. Memoria de largo plazo (IE3) — experiencias previas

Antes de auditar, `buscar_experiencias()` recupera casos parecidos de corridas
anteriores (colección Chroma `experiencias_auditoria`). Si acabas de correr los
ejemplos anteriores, aquí ya hay experiencias guardadas.

In [ ]:
estado = invocar_agente(
    "Quiero cambiar la caldera a gas de la sala de maquinas por una mas eficiente",
    categoria="gas",
    thread_id="demo-4",
)
print("Experiencias previas recuperadas:", len(estado.get("experiencias_previas", [])))
for e in estado.get("experiencias_previas", []):
    print("  -", e.get("texto", e))
print()
mostrar(estado, con_bitacora=False)

## 6. Bitácora y trazabilidad (IE6)

Toda decisión queda registrada en `bitacora` y en `data/agente_ejecuciones.jsonl`.

In [ ]:
import json
from pathlib import Path

bitacora = estado.get("bitacora", [])
for b in bitacora:
    print(f"[{b['nodo']}] {b['decision']}")

log = Path("data/agente_ejecuciones.jsonl")
if log.exists():
    lineas = log.read_text(encoding="utf-8").strip().splitlines()
    ultima = json.loads(lineas[-1])
    print(f"\nLog JSONL: {len(lineas)} ejecuciones registradas.")
    print("Ultima ejecucion -> thread:", ultima.get("thread_id"), "| tipo:", ultima.get("tipo_tarea"))
else:
    print("\nAun no existe data/agente_ejecuciones.jsonl (se crea con la primera ejecucion).")

## 7. Modo MCP — la tool `revisar_publicacion` (Evaluación N°1)

El mismo núcleo de auditoría, expuesto como herramienta MCP para Claude Desktop/Code.
Se puede invocar directamente en Python (devuelve un dict JSON-serializable).

In [ ]:
from src.agent.auditor import auditar_publicacion

resultado = auditar_publicacion(
    "Necesito reparar la caldera a gas de la sala de maquinas",
    "gas",
)
print("estado:", resultado["estado"])
print("alertas:")
for a in resultado["alertas"]:
    print(f"  - {a['fuente']} | {a['articulo']}: {a['mensaje'][:120]}...")
print("recomendaciones:", resultado.get("recomendaciones_generales", []))

---

**Qué demostrar con este notebook**

| Ejemplo | Qué evidencia |
|---|---|
| 1 | IE1 (herramientas ejecutadas con autonomía) e IE5 (plan priorizado) |
| 2 | IE7 (decisión adaptativa: publicación vaga → cuestionario `[COMPLETAR]`, sin inventar) |
| 3 | IE7 (ruta alterna `consulta_normativa` sin auditoría) |
| 4 | IE3 (memoria de corto plazo por `thread_id`) |
| 5 | IE3 (memoria de largo plazo: experiencias recuperadas) |
| 6 | IE6 (trazabilidad de decisiones) |
| 7 | IE1 (herramienta `revisar_publicacion` del MCP server) |

Más evidencia corrida contra la API real: `python -m scripts.demo_agente_ev2` (5 escenarios, `data/eval/ev2/`).